In [1]:
!pip install pandas scikit-learn gradio

In [3]:
# Import Pandas library
# 'pd' is the short name we use for Pandas

import pandas as pd

In [ ]:
# Import NumPy
import numpy as np

# Import TF-IDF Vectorizer
# We will use this later to convert text into numerical vectors
from sklearn.feature_extraction.text import TfidfVectorizer

# Import cosine similarity
# We will use this later to compare the similarity between questions
from sklearn.metrics.pairwise import cosine_similarity

# Import Gradio
# We will use this later to create the chatbot user interface
import gradio as gr

In [4]:
# Check whether the basic project setup is working

print("FAQ Chatbot setup completed successfully!")

FAQ Chatbot setup completed successfully!


In [10]:
# Create our own FAQ dataset for an E-commerce Customer Support Chatbot
# The dataset contains 20 frequently asked questions and their answers

faq_data = {
    
    # 20 frequently asked questions
    "question": [
        "What are your working hours?",
        "How can I contact customer support?",
        "What payment methods do you accept?",
        "How can I create an account?",
        "How can I reset my password?",
        "Do you provide refunds?",
        "How can I cancel my order?",
        "How long does delivery take?",
        "Where can I track my order?",
        "How can I contact you?",
        "How can I place an order?",
        "Can I change my delivery address?",
        "Is there a delivery charge?",
        "What should I do if my order is late?",
        "What should I do if I receive a damaged product?",
        "Can I return a product?",
        "How long does a refund take?",
        "Can I change my order after placing it?",
        "How do I check my order history?",
        "Do I need an account to place an order?"
    ],

    # Answers corresponding to the 20 questions above
    "answer": [
        "Our working hours are Monday to Friday, 9 AM to 6 PM.",
        "You can contact customer support through email or phone.",
        "We accept credit cards, debit cards and online payments.",
        "You can create an account by clicking the Sign Up button.",
        "You can reset your password using the Forgot Password option.",
        "Yes, refunds are available according to our refund policy.",
        "You can cancel your order from your account before it is shipped.",
        "Delivery usually takes 3 to 5 business days.",
        "You can track your order using the tracking option in your account.",
        "You can contact us through our official support email.",
        "Select the product you want, add it to your cart and complete the checkout process.",
        "Yes, you can change your delivery address before your order is shipped.",
        "Delivery charges may vary depending on the order and delivery location.",
        "Please check the order tracking information or contact customer support.",
        "Contact customer support and provide details about the damaged product.",
        "Yes, eligible products can be returned according to our return policy.",
        "Refund processing time may vary depending on the payment method.",
        "You may change your order before it is processed or shipped.",
        "You can view your previous orders from the Order History section of your account.",
        "An account may be required to place and manage your orders."
    ]
}

In [16]:
# Convert the FAQ dictionary into a Pandas DataFrame
# A DataFrame stores our questions and answers in table format

faq_df = pd.DataFrame(faq_data)

# Display the FAQ dataset
faq_df

,question,answer
0,What are your working hours?,"Our working hours are Monday to Friday, 9 AM t..."
1,How can I contact customer support?,You can contact customer support through email...
2,What payment methods do you accept?,"We accept credit cards, debit cards and online..."
3,How can I create an account?,You can create an account by clicking the Sign...
4,How can I reset my password?,You can reset your password using the Forgot P...
5,Do you provide refunds?,"Yes, refunds are available according to our re..."
6,How can I cancel my order?,You can cancel your order from your account be...
7,How long does delivery take?,Delivery usually takes 3 to 5 business days.
8,Where can I track my order?,You can track your order using the tracking op...
9,How can I contact you?,You can contact us through our official suppor...


In [13]:
# Count the number of FAQ records in our dataset

print("Number of FAQs:", len(faq_df))

Number of FAQs: 20


In [14]:
# Display the names of the columns in our FAQ dataset

print("Columns in the dataset:")
print(faq_df.columns)

Columns in the dataset:
Index(['question', 'answer'], dtype='object')


In [17]:
# Save our FAQ dataset as a CSV file
# index=False prevents Pandas from adding an extra index column

faq_df.to_csv("faq_dataset.csv", index=False)

print("FAQ dataset saved successfully!")

FAQ dataset saved successfully!


In [18]:
# Read the CSV file again to verify that it was saved correctly

test_df = pd.read_csv("faq_dataset.csv")

# Display the saved dataset

test_df

,question,answer
0,What are your working hours?,"Our working hours are Monday to Friday, 9 AM t..."
1,How can I contact customer support?,You can contact customer support through email...
2,What payment methods do you accept?,"We accept credit cards, debit cards and online..."
3,How can I create an account?,You can create an account by clicking the Sign...
4,How can I reset my password?,You can reset your password using the Forgot P...
5,Do you provide refunds?,"Yes, refunds are available according to our re..."
6,How can I cancel my order?,You can cancel your order from your account be...
7,How long does delivery take?,Delivery usually takes 3 to 5 business days.
8,Where can I track my order?,You can track your order using the tracking op...
9,How can I contact you?,You can contact us through our official suppor...


In [19]:
# Check whether any question or answer is missing

print("Missing values:")
print(faq_df.isnull().sum())

Missing values:
question    0
answer      0
dtype: int64


In [20]:
# Display only the questions from our dataset

print(faq_df["question"])

0                         What are your working hours?
1                  How can I contact customer support?
2                  What payment methods do you accept?
3                         How can I create an account?
4                         How can I reset my password?
5                              Do you provide refunds?
6                           How can I cancel my order?
7                         How long does delivery take?
8                          Where can I track my order?
9                               How can I contact you?
10                           How can I place an order?
11                   Can I change my delivery address?
12                         Is there a delivery charge?
13               What should I do if my order is late?
14    What should I do if I receive a damaged product?
15                             Can I return a product?
16                        How long does a refund take?
17             Can I change my order after placing it?
18        

In [21]:
# Convert all FAQ questions to lowercase
# This makes text comparison easier later

faq_df["clean_question"] = faq_df["question"].str.lower()
faq_df[["question", "clean_question"]]

,question,clean_question
0,What are your working hours?,what are your working hours?
1,How can I contact customer support?,how can i contact customer support?
2,What payment methods do you accept?,what payment methods do you accept?
3,How can I create an account?,how can i create an account?
4,How can I reset my password?,how can i reset my password?
5,Do you provide refunds?,do you provide refunds?
6,How can I cancel my order?,how can i cancel my order?
7,How long does delivery take?,how long does delivery take?
8,Where can I track my order?,where can i track my order?
9,How can I contact you?,how can i contact you?


In [22]:
# Remove unnecessary spaces from the beginning and end of questions
faq_df["clean_question"] = faq_df["clean_question"].str.strip()
faq_df["clean_question"]

0                         what are your working hours?
1                  how can i contact customer support?
2                  what payment methods do you accept?
3                         how can i create an account?
4                         how can i reset my password?
5                              do you provide refunds?
6                           how can i cancel my order?
7                         how long does delivery take?
8                          where can i track my order?
9                               how can i contact you?
10                           how can i place an order?
11                   can i change my delivery address?
12                         is there a delivery charge?
13               what should i do if my order is late?
14    what should i do if i receive a damaged product?
15                             can i return a product?
16                        how long does a refund take?
17             can i change my order after placing it?
18        

In [24]:
# Remove punctuation from the cleaned questions
# This makes text processing easier

faq_df["clean_question"] = faq_df["clean_question"].str.replace(
    r"[^\w\s]", 
    "", 
    regex=True
)

faq_df[["question", "clean_question"]]

,question,clean_question
0,What are your working hours?,what are your working hours
1,How can I contact customer support?,how can i contact customer support
2,What payment methods do you accept?,what payment methods do you accept
3,How can I create an account?,how can i create an account
4,How can I reset my password?,how can i reset my password
5,Do you provide refunds?,do you provide refunds
6,How can I cancel my order?,how can i cancel my order
7,How long does delivery take?,how long does delivery take
8,Where can I track my order?,where can i track my order
9,How can I contact you?,how can i contact you


In [25]:
# Check whether any answer is missing

print("Missing answers:", faq_df["answer"].isnull().sum())

Missing answers: 0


In [26]:
# Display the complete cleaned dataset

faq_df

,question,answer,clean_question
0,What are your working hours?,"Our working hours are Monday to Friday, 9 AM t...",what are your working hours
1,How can I contact customer support?,You can contact customer support through email...,how can i contact customer support
2,What payment methods do you accept?,"We accept credit cards, debit cards and online...",what payment methods do you accept
3,How can I create an account?,You can create an account by clicking the Sign...,how can i create an account
4,How can I reset my password?,You can reset your password using the Forgot P...,how can i reset my password
5,Do you provide refunds?,"Yes, refunds are available according to our re...",do you provide refunds
6,How can I cancel my order?,You can cancel your order from your account be...,how can i cancel my order
7,How long does delivery take?,Delivery usually takes 3 to 5 business days.,how long does delivery take
8,Where can I track my order?,You can track your order using the tracking op...,where can i track my order
9,How can I contact you?,You can contact us through our official suppor...,how can i contact you


In [27]:
# Save the cleaned FAQ dataset as a new CSV file

faq_df.to_csv("faq_dataset_cleaned.csv", index=False)

print("Cleaned FAQ dataset saved successfully!")

Cleaned FAQ dataset saved successfully!


In [28]:
import pandas as pd
faq_df = pd.read_csv("faq_dataset_cleaned.csv")
faq_df

,question,answer,clean_question
0,What are your working hours?,"Our working hours are Monday to Friday, 9 AM t...",what are your working hours
1,How can I contact customer support?,You can contact customer support through email...,how can i contact customer support
2,What payment methods do you accept?,"We accept credit cards, debit cards and online...",what payment methods do you accept
3,How can I create an account?,You can create an account by clicking the Sign...,how can i create an account
4,How can I reset my password?,You can reset your password using the Forgot P...,how can i reset my password
5,Do you provide refunds?,"Yes, refunds are available according to our re...",do you provide refunds
6,How can I cancel my order?,You can cancel your order from your account be...,how can i cancel my order
7,How long does delivery take?,Delivery usually takes 3 to 5 business days.,how long does delivery take
8,Where can I track my order?,You can track your order using the tracking op...,where can i track my order
9,How can I contact you?,You can contact us through our official suppor...,how can i contact you


In [30]:
# Import TF-IDF Vectorizer from Scikit-learn
# It converts text into numerical vectors
from sklearn.feature_extraction.text import TfidfVectorizer

# Create a TF-IDF Vectorizer object
vectorizer = TfidfVectorizer()

# Convert all cleaned FAQ questions into numerical TF-IDF vectors
tfidf_matrix = vectorizer.fit_transform(faq_df["clean_question"])

In [31]:
# Display the size of the TF-IDF matrix

print("TF-IDF Matrix Shape:", tfidf_matrix.shape)

TF-IDF Matrix Shape: (20, 52)


In [32]:
# Get all words/features learned by the TF-IDF vectorizer

words = vectorizer.get_feature_names_out()
print(words)

['accept' 'account' 'address' 'after' 'an' 'are' 'can' 'cancel' 'change'
 'charge' 'check' 'contact' 'create' 'customer' 'damaged' 'delivery' 'do'
 'does' 'history' 'hours' 'how' 'if' 'is' 'it' 'late' 'long' 'methods'
 'my' 'need' 'order' 'password' 'payment' 'place' 'placing' 'product'
 'provide' 'receive' 'refund' 'refunds' 'reset' 'return' 'should'
 'support' 'take' 'there' 'to' 'track' 'what' 'where' 'working' 'you'
 'your']


In [33]:
# Display the first FAQ question

print("Question:")
print(faq_df["clean_question"].iloc[0])

Question:
what are your working hours


In [34]:
# Display the numerical TF-IDF representation of the first question

print(tfidf_matrix[0].toarray())

[[0.         0.         0.         0.         0.         0.46994802
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.46994802 0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.34146076
  0.         0.46994802 0.         0.46994802]]


In [68]:
# Create a sample question from the user
user_question = ["When are you open"]

# Convert the user's question into a TF-IDF vector
user_vector = vectorizer.transform(user_question)
print(user_vector.toarray())


[[0.         0.         0.         0.         0.         0.78347027
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.62142927 0.        ]]


In [69]:
tfidf_matrix = vectorizer.fit_transform(faq_df["clean_question"])

user_vector = vectorizer.transform(user_question)

In [70]:
from sklearn.metrics.pairwise import cosine_similarity

# Compare the user's question vector with all FAQ question vectors
similarity_scores = cosine_similarity(
    user_vector,
    tfidf_matrix
)

# Display the similarity scores
print("Similarity Scores:")
print(similarity_scores)

Similarity Scores:
[[0.3681903  0.         0.23109693 0.         0.         0.28357491
  0.         0.         0.         0.35633855 0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.        ]]


In [41]:
# Find the position of the highest similarity score
best_match_index = similarity_scores.argmax()

print("Best Match Index:", best_match_index)

Best Match Index: 0


In [42]:
# Get the FAQ question that has the highest similarity
best_question = faq_df.iloc[best_match_index]["question"]

print("Best Matching Question:", best_question)

Best Matching Question: What are your working hours?


In [43]:
# Get the answer corresponding to the best matching question
best_answer = faq_df.iloc[best_match_index]["answer"]

print("Chatbot Answer:", best_answer)

Chatbot Answer: Our working hours are Monday to Friday, 9 AM to 6 PM.


In [44]:
# Get the highest similarity score
best_score = similarity_scores[0][best_match_index]

print("Similarity Score:", best_score)

Similarity Score: 0.36819029915834145


In [106]:
import re

def chatbot(user_question):

    # Step 1: Clean user question

    clean_question = user_question.lower().strip()

    # Remove punctuation
    clean_question = re.sub(
        r"[^\w\s]",
        "",
        clean_question
    )

    # Step 2: Check whether question is related
    # to our e-commerce chatbot

    ecommerce_keywords = {
        "order", "orders",
        "delivery", "deliver",
        "shipping", "shipped",
        "payment", "payments",
        "refund", "refunds",
        "return", "returns",
        "product", "products",
        "account", "password",
        "support", "customer",
        "address",
        "tracking", "track",
        "purchase", "buy",
        "shopping",
        "cancel",
        "working", "hours",
        "open",
        "contact"
    }

    # Split the user question into words
    user_words = set(clean_question.split())

    # Check for common e-commerce keywords
    topic_words = user_words.intersection(ecommerce_keywords)

    # If no e-commerce word is found, reject the question
    if len(topic_words) == 0:
        return (
            "Sorry, I don't understand your question. "
            "Please ask about our online shopping services."
        )

    # Step 3: Convert question to TF-IDF

    user_vector = vectorizer.transform(
        [clean_question]
    )

    # Step 4: Calculate cosine similarity

    similarity_scores = cosine_similarity(
        user_vector,
        tfidf_matrix
    )

    # Step 5: Find best matching FAQ

    best_match_index = similarity_scores.argmax()

    best_score = similarity_scores[0][best_match_index]

    # Step 6: Similarity threshold

    threshold = 0.35

    if best_score < threshold:
        return (
            "Sorry, I don't understand your question. "
            "Please ask about our online shopping services."
        )

    # Step 7: Get the answer

    best_answer = faq_df.iloc[best_match_index]["answer"]

    return best_answer

In [107]:
print(chatbot("What is the capital of India?"))

Sorry, I don't understand your question. Please ask about our online shopping services.


In [108]:
print(chatbot("How can I track my order?"))

You can track your order using the tracking option in your account.


In [109]:
print(chatbot("Can I return my product?"))

Yes, eligible products can be returned according to our return policy.


In [110]:
print(chatbot("When are you open?"))

Our working hours are Monday to Friday, 9 AM to 6 PM.


In [122]:
# Improved chatbot with greetings

def final_chatbot(user_question):

    # Convert the question to lowercase
    question = user_question.lower().strip()

    # Handle greetings
    greetings = ["hi", "hello", "hey", "good morning", "good evening"]

    if question in greetings:
        return (
            "Hello! 👋 Welcome to our E-Commerce Customer Support. "
            "How can I help you today?"
        )

    # Handle goodbye
    goodbyes = ["bye", "goodbye", "thank you", "thanks"]

    if question in goodbyes:
        return (
            "You're welcome! 😊 "
            "Thank you for using our E-Commerce FAQ Chatbot."
        )

    # Use the existing chatbot for other questions
    return chatbot(user_question)

In [123]:
import gradio as gr

# Connect the final chatbot to Gradio
def respond(message, history):

    # Check for empty message
    if not message or message.strip() == "":
        return "Please enter a question."

    # Get chatbot response
    return final_chatbot(message)


# Create the final interface
demo = gr.ChatInterface(
    fn=respond,
    title="🛒 E-Commerce FAQ Chatbot",

    description="""
    🤖 Welcome to our E-Commerce Customer Support!

    Ask me about:
    • Orders
    • Payments
    • Delivery
    • Returns and refunds
    • Accounts and passwords
    """,

    examples=[
        "How can I track my order?",
        "Can I return a product?",
        "How can I reset my password?",
        "What payment methods do you accept?",
        "How long does delivery take?"
    ]
)

demo.launch()

* Running on local URL:  http://127.0.0.1:7867
* To create a public link, set `share=True` in `launch()`.
